# 03 – PostgreSQL database

**Goal:** Store the integrated tables in PostgreSQL (container in the Codespace) and query them from Python with SQL, including joins and aggregations.
**Input:** `data/interim/*.parquet` (notebook 02) · **Output:** filled database tables
**Assessment items:** P3 (database with SQL queries from Python), B2 (PostgreSQL instead of SQLite, joins and aggregations)

Schema: `sql/schema.sql` – primary and foreign keys connect `circuits` → `races` → `results` / `laps` / `weather` / `pairs`, and `drivers` → `driver_map` / `results` / `laps` / `pairs`.

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [ ]:
from sqlalchemy import text

from src.config import INTERIM_DIR
from src.db import TABLE_ORDER, get_engine, load_tables, reset_schema, table_counts

engine = get_engine()
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one().split(",")[0])

## 1. Create schema and load the tables

`reset_schema` drops and recreates all project tables, so the notebook always produces the same database state.

In [ ]:
reset_schema(engine)
tables = {name: pd.read_parquet(INTERIM_DIR / f"{name}.parquet")
          for name in TABLE_ORDER if name != "pairs"}
load_tables(engine, tables)
pd.Series(table_counts(engine)).to_frame("rows")

## 2. SQL queries from Python

**Query 1 – races and laps per era and circuit type** (join `races` × `circuits`, aggregation)

In [ ]:
pd.read_sql("""
    SELECT r.era,
           c.circuit_type,
           COUNT(*)                 AS races,
           ROUND(AVG(r.total_laps)) AS avg_laps,
           ROUND(AVG(c.length_km), 3) AS avg_length_km
    FROM races r
    JOIN circuits c USING (circuit_id)
    GROUP BY r.era, c.circuit_type
    ORDER BY r.era, races DESC
""", engine)

**Query 2 – pit stops and neutralised laps per race** (join `laps` × `races` × `circuits`, conditional aggregation)

In [ ]:
per_race = pd.read_sql("""
    SELECT r.season, r.round, r.race_name, c.circuit_type,
           COUNT(*)                                       AS laps,
           COUNT(*) FILTER (WHERE l.pit_in)               AS pit_stops,
           ROUND(100.0 * AVG((l.track_status ~ '[4567]')::int), 1) AS pct_laps_sc_vsc_red
    FROM laps l
    JOIN races r    USING (race_id)
    JOIN circuits c USING (circuit_id)
    GROUP BY r.season, r.round, r.race_name, c.circuit_type
    ORDER BY r.season, r.round
""", engine)
per_race.head(10)

In [ ]:
per_race.groupby("season")[["pit_stops", "pct_laps_sc_vsc_red"]].mean().round(1)

**Query 3 – positions gained from grid to finish per team and season** (join `results` × `races`, aggregation)

In [ ]:
pd.read_sql("""
    SELECT r.season, res.team,
           COUNT(*) AS starts,
           ROUND(AVG(res.grid - res.finish_pos), 2) AS avg_positions_gained
    FROM results res
    JOIN races r USING (race_id)
    WHERE res.grid > 0 AND res.finish_pos IS NOT NULL
    GROUP BY r.season, res.team
    ORDER BY r.season, avg_positions_gained DESC
""", engine)